# FastWindTerrain from Python

A mass-consistent wind solver driven entirely from Python: no inputs
file, no shelling out, no reading results back off disk.

The tour goes grid → terrain → solve → fields → dataset. The last step is
what the bindings exist for: many cases in one process, on one fixed
grid, stacked into an array a neural operator can train on.

Two things to know before you start.

**AMReX's lifecycle is process-global.** `initialize()` and `finalize()`
happen once per process and cannot be repeated, so they are explicit
rather than implicit on import. In a script you would write
`with fwt.session():`; a notebook's cells do not nest inside a `with`,
so this one initializes at the top and finalizes at the bottom. If you
restart the kernel, start again from that first cell.

**Nothing here reads ParmParse.** Every case below is a dict. That is not
cosmetic: ParmParse is global and persists, so a case that omits a
parameter would otherwise inherit whatever an earlier case set — which
in a generation loop is silent corruption spread across a dataset.

In [ ]:
import os
import tempfile
from pathlib import Path

import numpy as np

import fastwindterrain as fwt
from fastwindterrain import dataset

print("fastwindterrain", fwt.__version__, "| AMReX", fwt.amrex_version())

# Run from anywhere in the repo.
here = Path.cwd()
repo = next((p for p in [here, *here.parents] if (p / "regtests").is_dir()), here)
TERRAIN = repo / "regtests" / "phase8_diagnostics_output" / "terrain_hill.csv"

# Everything this notebook writes goes here and nowhere else.
WORK = Path(tempfile.mkdtemp(prefix="fwt-quickstart-"))
print("scratch:", WORK)

In [ ]:
fwt.initialize([])          # once per process; finalize() is the last cell

## 1. The grid

Cartesian in x and y, stretched in z: cells start at `dz0` and grow by
`stretching_ratio`, so the boundary layer is resolved without paying for
that resolution all the way to the domain top.

The column height is determined by `n_cell[2]`, `dz0` and the ratio.
FastWindTerrain will not quietly reinterpret what you asked for: a column
that overshoots the requested `prob_hi` warns (and you can promote that
to an exception), and one that undershoots raises.

In [ ]:
GRID = {
    "n_cell": (24, 24, 40),
    "prob_lo": (0.0, 0.0, 0.0),
    "prob_hi": (1000.0, 1000.0, 483.19909696997223),
    "dz0": 4.0,
    "stretching_ratio": 1.05,
    "max_grid_size": 16,
}

g = fwt.Grid(GRID)
print(g)
print("first three cells:", np.diff(g.z_face)[:3].round(3), "m")
print("last three cells: ", np.diff(g.z_face)[-3:].round(3), "m")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4, 3.2))
ax.plot(np.diff(g.z_face), g.z_cc, marker=".", lw=1)
ax.set_xlabel("cell height dz [m]")
ax.set_ylabel("height z [m]")
ax.set_title("Vertical stretching")
fig.tight_layout()

## 2. Terrain

Terrain arrives as an `(n, 3)` point cloud and is interpolated onto grid
columns by inverse distance weighting. It is represented as an immersed
boundary: cells below the surface are marked solid in `mask` and take no
part in the solve.

Handing in a numpy array and pointing at the CSV those numbers came from
give identical results — the array path is the same code, not a second
one.

In [ ]:
points = np.loadtxt(TERRAIN, delimiter=",", comments="#", skiprows=5)
print("point cloud:", points.shape)

t = fwt.Terrain(g, {"points": points})
print(t)
print(f"solid cells: {t.n_solid} of {t.n_total} "
      f"({100 * t.n_solid / t.n_total:.1f}%)")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8, 3.2))

im = a1.imshow(t.z_terrain[0], origin="lower", cmap="terrain")
a1.set_title("terrain elevation [m]")
fig.colorbar(im, ax=a1, shrink=0.85)

# A vertical slice through the middle of the domain: solid below, fluid
# above.
a2.imshow(t.mask[:, t.mask.shape[1] // 2, :], origin="lower",
          aspect="auto", cmap="gray_r")
a2.set_title("mask, j = mid (1 = solid)")
a2.set_xlabel("i")
a2.set_ylabel("k")
fig.tight_layout()

## 3. Solving, one pass at a time

`Solver.run()` does the whole thing. The stages are also separate, which
is what makes a notebook worth using: `project_once()` runs a single
projection pass and returns the MLMG residual, so you can watch the
divergence come down rather than be told that it did.

`max_divergence_fe` is the divergence in the norm the projection actually
controls — the number that measures whether a pass helped.

In [ ]:
CASE = {
    "grid": GRID,
    "terrain": {"points": points},
    "inflow": {"mode": "powerlaw", "u_ref": 8.0, "v_ref": 6.0},
    "anisotropy": {"enable": True},
    "obrien": {"enable": True},
    "poisson": {"alpha_v": 0.5, "n_projections": 6},
}

s = fwt.Solver(CASE)
s.setup()

divergence = [s.max_divergence_fe]
for step in range(6):
    residual = s.project_once()
    divergence.append(s.max_divergence_fe)
    print(f"pass {step + 1}: max|div| = {divergence[-1]:.6g}   "
          f"MLMG residual {residual:.2e} in {s.solve_iterations} iterations")

s.diagnose()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.2))
ax.semilogy(divergence, marker="o")
ax.set_xlabel("projection pass")
ax.set_ylabel("max |div(u)|  [1/s]")
ax.set_title("The projection converging")
ax.grid(alpha=0.3)
fig.tight_layout()

## 4. Fields, without a file

`fields()` returns every output field as a dict of numpy arrays — the
same object the plotfile and ascii writers are handed. One gather, three
consumers, so what you get here is exactly what the file would have
contained.

Arrays are `(nz, ny, nx)`, or `(ncomp, nz, ny, nx)` where there is more
than one component. Channels-first is AMReX's own memory order, so a
gather is a memcpy rather than a transpose, and it is what PyTorch's
`conv3d` wants.

In [ ]:
f = s.fields()
print(len(f), "fields:", ", ".join(sorted(f)))
print("u:", f["u"].shape, f["u"].dtype)

u, w, mask = f["u"], f["w"], f["mask"]
print(f"|U| max = {np.sqrt(f['u']**2 + f['v']**2 + f['w']**2).max():.3f} m/s")

In [ ]:
j = u.shape[1] // 2
fig, (a1, a2) = plt.subplots(2, 1, figsize=(7, 5), sharex=True)

for ax, field, title, cmap in ((a1, u, "u [m/s]", "viridis"),
                               (a2, w, "w [m/s]", "coolwarm")):
    plot = np.ma.masked_where(mask[:, j, :] > 0.5, field[:, j, :])
    lim = np.abs(plot).max()
    im = ax.pcolormesh(np.arange(plot.shape[1]), g.z_cc, plot, cmap=cmap,
                       vmin=-lim if cmap == "coolwarm" else None,
                       vmax=lim if cmap == "coolwarm" else None,
                       shading="nearest")
    ax.plot(np.arange(plot.shape[1]), t.z_terrain[0, j, :], "k", lw=1.5)
    ax.set_ylabel("z [m]")
    ax.set_title(title)
    fig.colorbar(im, ax=ax)

a2.set_xlabel("i")
fig.suptitle("Vertical slice through the hill")
fig.tight_layout()

## 5. Generating a dataset

Now the reason for all of it. `dataset.sweep` builds configs, and
`dataset.generate` solves each one in this same process and stacks the
results.

The grid is held fixed and everything else is free to vary — a neural
operator needs one tensor shape for every sample. That is enforced, not
assumed: sweeping a grid parameter raises, and so does a config whose
grid section differs from the first one's.

Pick the fields you need. All seventeen is usually about three times the
data a surrogate wants, and `dtype="float32"` halves it again (the solve
is always double; the cast happens on the way out).

In [ ]:
configs = dataset.sweep(CASE, {"inflow.u_ref": [4.0, 8.0, 12.0],
                               "inflow.v_ref": [0.0, 6.0]})
print(len(configs), "cases")

manifest = dataset.generate(
    configs, str(WORK / "wind.npz"),
    fields=["u", "v", "w", "mask", "terrain_z"],
    dtype="float32", progress=True, seed=0,
)
print("wrote", manifest["files"])

In [ ]:
arrays, manifest = dataset.load(str(WORK / "wind.npz"))

for name, a in arrays.items():
    print(f"{name:12s} {str(a.shape):22s} {a.dtype}")

print()
print("sample 0 was solved with u_ref =",
      manifest["configs"][0]["inflow"]["u_ref"],
      "v_ref =", manifest["configs"][0]["inflow"]["v_ref"])
print("the vertical coordinate travels with the data:",
      arrays["grid_z_cc"].shape)

The terrain point cloud is not written once per sample — the manifest
keeps a shape/dtype/SHA-256 stamp instead, and the terrain itself is
recoverable from `terrain_z` and `mask`.

For a dataset larger than memory, either pass `shard_size=` (which writes
a directory of `shard_*.npz` and keeps peak memory to one shard) or use
`dataset.iter_samples`, which yields one solved case at a time and lets
you write whatever format you like.

In [ ]:
stamp = manifest["configs"][0]["terrain"]["points"]["__ndarray__"]
print("terrain stamp:", stamp["shape"], stamp["dtype"], stamp["sha256"][:16])

# Streaming, for when the stack will not fit:
for sample in dataset.iter_samples(configs[:2], fields=["u"]):
    print(f"sample {sample.index}: u max {sample.arrays['u'].max():.3f} m/s, "
          f"{sample.info['solve_iterations']} MLMG iterations")

## Cleaning up

`finalize()` ends the AMReX session. It is not optional: skipping it
leaves the process unable to initialize again, so re-running this
notebook would need a kernel restart.

In [ ]:
import shutil

fwt.finalize()
shutil.rmtree(WORK, ignore_errors=True)
print("done")